In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [130]:
import math

from typing import List

class SiLU(nn.Module):
    def forward(self, x):
        return x * F.sigmoid(x)

class SwiGLU(nn.Module):
    def __init__(self, emb_size: int, dropout: float =0.1):
        super().__init__()
        self.emb_size = emb_size
        self.dropout = dropout

        self.gate = nn.Linear(self.emb_size, 4*self.emb_size)
        self.up = nn.Linear(self.emb_size, 4*self.emb_size)
        self.down = nn.Linear(4*self.emb_size, self.emb_size)
        self.silu = SiLU()
        self.drpt = nn.Dropout(dropout)

    def forward(self, x):
        return self.drpt(self.down(self.up(x) * self.silu(self.gate(x))))

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps=1e-6):
        super().__init__()
        self.dim = dim
        self.eps = eps

        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        a = torch.sqrt(torch.mean(x ** 2, 2,keepdim=True) + self.eps)
        b = x/a
        return torch.mul(self.weight , b)

class RoPE(nn.Module):
    def __init__(self, head_size: int, max_seq_len: int, base: int =10_000):
        super().__init__()
        self.head_size = head_size
        self.max_seq_len = max_seq_len
        self.base = base

        theta = 1 / torch.Tensor([self.base]) ** (torch.Tensor([i for i in range(0, self.head_size, 2)]) / self.head_size)
        positional_indexes = torch.arange(self.max_seq_len, dtype=torch.float32)
        frequency_matrix = positional_indexes.reshape(positional_indexes.shape[0], 1) * theta
        self.cosinus_matrix = torch.cos(frequency_matrix)
        self.sinus_matrix = torch.sin(frequency_matrix)

    def forward(self, x, start_pos=0):
        batch_size, num_heads, seq_len, head_size = x.shape

        local_cosinus_matrix = self.cosinus_matrix[start_pos:start_pos + seq_len,  :]
        local_sinus_matrix = self.sinus_matrix[start_pos:start_pos + seq_len,  :]

        even = x[:,:,:,::2]
        odd = x[:,:,:,1::2]

        return_tensor = torch.zeros_like(x)

        return_tensor[:,:,:, ::2] = even * local_cosinus_matrix - odd * local_sinus_matrix
        return_tensor[:,:,:, 1::2] = odd * local_cosinus_matrix + even * local_sinus_matrix
                
        return return_tensor

class TokenEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, emb_size: int):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, emb_size)

    def forward(self, x: torch.Tensor):
        return self.embeddings(x)



class GroupedQueryAttention(nn.Module):
    def __init__(self, num_q_heads: int, num_kv_heads, emb_size: int,
                head_size: int, max_seq_len: int, rope: RoPE, window_size: int, dropout: float = 0.1):
        super().__init__()
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len
        self.dropout = dropout
        self.rope = rope
        self.window_size = window_size

        self.W_q = nn.Linear(self.emb_size, self.head_size*self.num_q_heads)
        self.W_k = nn.Linear(self.emb_size, self.head_size*self.num_kv_heads)
        self.W_v = nn.Linear(self.emb_size, self.head_size*self.num_kv_heads)
        
        self.mask = (torch.tril(torch.ones(self.max_seq_len, self.max_seq_len)) > 0) \
                    &(~(torch.tril(torch.ones(self.max_seq_len, self.max_seq_len), diagonal= - self.window_size - 1) > 0 ))

        
        self.otpt = nn.Linear(self.head_size * self.num_q_heads, self.emb_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, use_cache: bool=True, cache=None):
        batch_size, seq_len, emb_size = x.shape
                
        query = self.W_q(x)
        key = self.W_k(x)
        value = self.W_v(x)

        query = query.reshape(batch_size, seq_len, self.num_q_heads, self.head_size).permute(0,2,1,3)
        key = key.reshape(batch_size, seq_len, self.num_kv_heads, self.head_size).permute(0,2,1,3)
        value = value.reshape(batch_size, seq_len, self.num_kv_heads, self.head_size).permute(0,2,1,3)

        
        if cache is not None:
            start_pos = cache[0].shape[2]
            key = self.rope(key, start_pos)
            query = self.rope(query, start_pos)
        else:
            key = self.rope(key)
            query = self.rope(query)

        if cache is not None:
            key = torch.cat([cache[0], key], 2)
        if cache is not None:
            value = torch.cat([cache[1], value], 2)

        if use_cache == True:
            new_cache = (key, value) 
        num_repeat = self.num_q_heads // self.num_kv_heads
        
        
        new_key = torch.zeros(batch_size, self.num_q_heads, key.shape[2], self.head_size)
        new_value = torch.zeros(batch_size, self.num_q_heads, value.shape[2], self.head_size)
        for i in range(self.num_kv_heads):
            for j in range(num_repeat):
                new_key[:, i*num_repeat + j] = key[:, i]
                new_value[:, i*num_repeat + j] = value[:, i]

        attention_matrix : torch.Tensor = query @ new_key.transpose(2,3) / math.sqrt(self.head_size)

        if cache is None:
            mask = self.mask[:seq_len, :seq_len] == 0
            attention_matrix = attention_matrix.masked_fill(mask, float('-inf'))

        attention_matrix = torch.softmax(attention_matrix, 3) 
        x = attention_matrix @ new_value

        x = x.permute(0,2,1,3).reshape(batch_size, seq_len, self.num_q_heads * self.head_size)
        tensor = self.otpt(x)
        tensor = self.dropout(tensor)

        if use_cache:
            key = key[:,:,-self.window_size:,:]
            value = value[:,:,-self.window_size:,:]
            return (tensor, (key, value))

        return (tensor, None)


class Decoder(nn.Module):
    def __init__(self, num_q_heads: int, num_kv_heads: int, emb_size: int,
                head_size: int, max_seq_len: int, rope: RoPE, window_size: int,  dropout: float = 0.1):
        super().__init__()
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len
        self.dropout = dropout
        self.window_size = window_size

        self.multi_head_attention = GroupedQueryAttention(self.num_q_heads, self.num_kv_heads, self.emb_size, self.head_size, self.max_seq_len, rope, self.window_size, self.dropout)
        self.feed_forward = SwiGLU(self.emb_size, self.dropout)
        self.first_norm = RMSNorm(self.emb_size)
        self.second_norm = RMSNorm(self.emb_size)

    def forward(self, x: torch.Tensor, use_cache = True, cache = None):
        tensor = self.first_norm(x)
        tensor, cache = self.multi_head_attention(tensor, use_cache, cache)
        tensor = x + tensor
        tensor = tensor + self.feed_forward(self.second_norm(tensor))
        if use_cache:
            return (tensor, cache)

        return (tensor, None)


class Mistral(nn.Module):
    def __init__(self, vocab_size: int, max_seq_len: int, emb_size: int,
                num_q_heads: int, num_kv_heads, head_size: int, num_layers: int,
                dropout: float = 0.1, window_size: int = 4096, device: str = "cpu"):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.emb_size = emb_size
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_size = head_size
        self.num_layers = num_layers
        self.dropout = dropout
        self.device = device
        self.window_size = window_size

        self.token_embeddings = TokenEmbeddings(self.vocab_size, self.emb_size)
        self.rope = RoPE(self.head_size, self.max_seq_len)
        #self.positional_embeddings = PositionalEmbeddings(self.max_seq_len, self.emb_size)
        self.drpt = nn.Dropout(self.dropout)
        self.decoders = [Decoder(self.num_q_heads, self.num_kv_heads, self.emb_size, self.head_size, self.max_seq_len, self.rope, self.window_size, self.dropout) for i in range(self.num_layers)]
        self.linear = nn.Linear(self.emb_size, self.vocab_size)
        self.norm = RMSNorm(self.emb_size)

    def forward(self, x: torch.Tensor, use_cache=True, cache=None):
        tensor = self.token_embeddings(x)
        tensor = self.drpt(tensor)
        new_caches = []
        if cache is not None:
            for ch, decoder in zip(cache,self.decoders):
                tensor, cache_decoder = decoder(tensor, use_cache, ch)
                new_caches.append(cache_decoder)
        else:
            for decoder in self.decoders:
                tensor, cache_decoder = decoder(tensor, use_cache, cache)
                new_caches.append(cache_decoder)
        tensor = self.norm(tensor)
        tensor = self.linear(tensor)
        if use_cache:
            return (tensor, new_caches)
        
        return (tensor, None) 
        
        
    def generate(self, x: torch.Tensor, max_new_tokens: int, do_sample: bool, temperature: float = 1.0,
                top_k: int = None, top_p: float = None, use_cache=True):

        for i in range(max_new_tokens):
            if i == 0:
                x_hat = x[:, :]
            else:
                x_hat = x[:, -1:]
            if i == 0:
                logits, cache = self.forward(x_hat, use_cache=use_cache, cache=None)
            else:
                logits, cache = self.forward(x_hat, use_cache=use_cache, cache=cache)
            logits = logits[:, -1, :]
            logits /= temperature
            if not do_sample:
                probas = logits.softmax(1)
                new_tokens = torch.argmax(probas, dim=1)
            else:
                if top_k is not None:
                    new_logits = []
                    for j in range(logits.shape[0]):
                        logits_j = logits[j,:]
                        topk_values, topk_indicies = torch.topk(logits_j, top_k)
                        mask = torch.ones_like(logits_j) == 1
                        mask[topk_indicies] = 0
                        logits_j.masked_fill_(mask, float('-inf'))
                        new_logits.append(logits_j)
                    logits = torch.stack(new_logits,)
                if top_p is not None:
                    new_logits = []
                    for j in range(logits.shape[0]):
                        logits_j = logits[j,:]
                        probabilities = logits_j.softmax(0)
                        
                        max_probabilities = sorted(probabilities.detach().numpy(),reverse=True)

                        cum_proba = max_probabilities[0]
                        min_proba = max_probabilities[0]
                        m = 1
                        while m < len(max_probabilities):
                            cum_proba += max_probabilities[m]
                            if cum_proba < top_p:
                                min_proba = max_probabilities[m]
                            else:
                                break
                            m += 1
                        mask = probabilities  < min_proba

                        logits_j.masked_fill_(mask, float('-inf'))
                        new_logits.append(logits_j)
                    logits = torch.stack(new_logits,)
                        
                probas = logits.softmax(1)
                new_tokens = torch.multinomial(probas, 1)
            new_tokens = torch.reshape(new_tokens, (new_tokens.shape[0], 1))
            x = torch.cat([x, new_tokens], dim=1)
        return x

    def save(self, path):
        torch.save({
            'model_state_dict': self.state_dict(),
            'vocab_size': self.vocab_size,
            'max_seq_len': self.max_seq_len,
            'emb_size': self.emb_size,
            'num_heads': self.num_heads,
            'head_size': self.head_size,
            'num_layers': self.num_layers
        }, path)
        
    @classmethod
    def load(cls, path, device):
        checkpoint = torch.load(path, map_location=device)
        model = cls(
            vocab_size=checkpoint['vocab_size'],
            max_seq_len=checkpoint['max_seq_len'],
            emb_size=checkpoint['emb_size'],
            num_heads=checkpoint['num_heads'],
            head_size=checkpoint['head_size'],
            num_layers=checkpoint['num_layers']
        )
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
        return model
    
    def fit(self, train_loader , valid_loader, 
           num_epoch: int, learning_rate: float,
           use_cache=False):

        self.to(self.device)

        criterion = nn.CrossEntropyLoss()

        adam = torch.optim.Adam(self.parameters(), lr=learning_rate)

        for epoch in range(num_epoch):
            self.train()

            for inputs, targets in train_loader:
                batch_size, seq_len = inputs.shape
                outputs, cache = self.forward(inputs, use_cache=use_cache)

                outputs = outputs.reshape(batch_size * seq_len, self.vocab_size)
                targets = targets.reshape(batch_size * seq_len)

                self.loss = criterion(outputs, targets)

                adam.zero_grad()
                self.loss.backward()

                adam.step()

            self.eval()
    
            with torch.no_grad():
                for inputs, targets in valid_loader:
                    
                    batch_size, seq_len = inputs.shape
                    outputs, cache = self.forward(inputs, use_cache=use_cache)
    
                    outputs = outputs.reshape(batch_size * seq_len, self.vocab_size)
                    targets = targets.reshape(batch_size * seq_len)
    
                    self.loss = criterion(outputs, targets)


In [131]:
GroupedQueryAttention(2, 1, 3, 4, 5, RoPE(4, 5), 2)\
.forward(torch.Tensor([[[1,2,3],[3,4,5]],[[6,5,4],[3,2,1]]]))



(tensor([[[-0.1064,  0.0000, -0.0169],
          [ 0.3428,  1.3307,  0.0563]],
 
         [[ 1.1145,  1.8926,  2.0396],
          [ 1.1659,  1.5390,  1.2705]]], grad_fn=<MulBackward0>),
 (tensor([[[[ 1.2577, -0.2569, -0.5381, -2.0796],
            [ 2.4177,  1.8989, -1.0594, -3.2966]]],
  
  
          [[[ 3.7176, -2.9672, -2.2638, -2.4388],
            [ 2.2225,  0.0561, -1.4261, -0.6438]]]], grad_fn=<SliceBackward0>),
  tensor([[[[-0.4884, -0.2408, -0.6734,  0.5875],
            [-1.8855, -0.2423, -0.7032,  0.3851]]],
  
  
          [[[-3.3647,  1.3511, -0.0296, -0.9253],
            [-1.2690,  1.3533,  0.0150, -0.6218]]]], grad_fn=<SliceBackward0>)))

In [119]:
torch.Tensor([[[ 1,  2,  3,  4,  5,  6],
  [ 7,  8,  9, 10, 11, 12],
  [13, 14, 15, 16, 17, 18],
  [19, 20, 21, 22, 23, 24]]]).reshape(1, 4,2,3).permute(0,2,1,3)

tensor([[[[ 1.,  2.,  3.],
          [ 7.,  8.,  9.],
          [13., 14., 15.],
          [19., 20., 21.]],

         [[ 4.,  5.,  6.],
          [10., 11., 12.],
          [16., 17., 18.],
          [22., 23., 24.]]]])

In [120]:
Mistral(10, 15,5, 1,1, 8,1).forward(torch.LongTensor([[5,4,3,2,1],[2,3,4,5,6]]))[0]



TypeError: tril(): argument 'diagonal' must be int, not float

In [121]:
~torch.tril(torch.ones(5, 5, dtype= torch.bool), diagonal=-2 - 1)

tensor([[ True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True],
        [False,  True,  True,  True,  True],
        [False, False,  True,  True,  True]])